# <span style="color:blue">2. Wave Equation  </span>

This notebook demonstrates how the 1D wave equation can be solved numerically using finite differences, and how the resulting scheme can be interpreted using linear algebra.

## Contents
1. Introduction  
2. Mathematical model
3. Finite Difference Discretization  
4. Simulation and Animation  
5. Matrix Formulation  
6. Eigenmodes of the Discrete Laplacian  
7. Evolution of a Single Eigenmode  


👉 **Eigenvectors of the discrete operator correspond to standing wave modes of the system.**


---
## 1. Introduction
### Connection to the Heat Equation Project

This example builds directly on ideas developed in the **Heat Equation project**.

In that project we studied the diffusion equation

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}.
$$

In the present example we study the **wave equation**

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}.
$$

Notice that both equations contain the **same spatial operator**

$$
\frac{\partial^2}{\partial x^2},
$$

which leads to the same discrete Laplacian structure used in the Heat Equation notebooks.

👉  The key difference is the **time derivative**:

| Equation | Physical behavior |
|---|---|
| Heat equation | diffusion and smoothing |
| Wave equation | propagation and oscillation |


---
### Wave equation phenomena

The **wave equation** describes oscillatory phenomena such as

- vibrating strings
- sound waves
- water waves
- electromagnetic waves.

Unlike diffusion processes, wave motion **transports disturbances through space** instead of smoothing them out.

Here we study

**1. How does a localized displacement evolve under the wave equation?**\
**2. Wave equation eigenmodes**



---

## 2. Mathematical Model

The one‑dimensional wave equation is

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}
$$

where

- $u(x,t)$ is the displacement
- $c$ is the wave speed
- $x$ is the spatial coordinate
- $t$ is time.

### Boundary conditions

For a string fixed at both ends

$$
u(0,t) = 0, \qquad u(L,t) = 0.
$$

### Initial conditions

We specify

$$
u(x,0) = u_0(x)
$$

and

$$
\frac{\partial u}{\partial t}(x,0) = v_0(x).
$$


---

## 3. Finite Difference Discretization

Space grid

$$
x_i = i h
$$

where $h$ is the spatial grid spacing (same notation as in the Heat Equation notebooks).

When spatial derivative approximation

$$
\frac{\partial^2 u}{\partial x^2}
\approx
\frac{u_{i-1}-2u_i+u_{i+1}}{h^2}
$$

and time derivative approximation

$$
\frac{\partial^2 u}{\partial t^2}
\approx
\frac{u_i^{n+1}-2u_i^n+u_i^{n-1}}{\Delta t^2}.
$$

are placed in the wave equation, we get the update rule

$$
u_i^{n+1} =
2u_i^n - u_i^{n-1}
+ \frac{c^2 \Delta t^2}{h^2}(u_{i-1}^n -2u_i^n + u_{i+1}^n).
$$

👉 **We approximate derivatives using neighboring grid points**

---
## 4. Simulation and Animation

In the next simulation the initial condition is a Gaussian bump. The code decomposes it implicitly into discrete eigenmodes. Each mode oscillates with its own frequency, and the animation shows the superposition of all these oscillations.

That is why the wave spreads, reflects, and forms complex patterns—yet the underlying evolution is just a sum of simple harmonic motions.

👉 **Each eigenmode evolves independently, so the full solution is a superposition of simple oscillations.**

In [ ]:
#  Wave propagation 

%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Physical parameters
L = 1.0
T = 1.0
c = 1.0

# Numerical parameters
Nx = 100
h = L / Nx
dt = 0.5 * h / c
Nt = int(T / dt)

x = np.linspace(0, L, Nx+1)

# Initial conditions
u0 = np.exp(-100*(x-0.5)**2)
v0 = np.zeros_like(x)

# Initialize time levels
u_prev = u0.copy()
u = u0 + dt*v0
u_next = np.zeros_like(u)

# Store frames for animation
frames = [u.copy()]                       # makes a deep copy of the current solution array u

for n in range(Nt):

    for i in range(1, Nx):
        u_next[i] = (
            2*u[i] - u_prev[i]
            + (c**2 * dt**2 / h**2) * (u[i-1] - 2*u[i] + u[i+1])
        )

    if n % 5 == 0:                        # store every 5th frame to keep animation light
        frames.append(u.copy())

    u_prev[:] = u
    u[:] = u_next


# Create animation
fig, ax = plt.subplots()
ax.set_autoscale_on(False)                # for increasing speed: “The axes never change — only update the line.”

line, = ax.plot(x, frames[0])

ax.set_xlim(0, L)
ax.set_ylim(-1.1*np.max(u0), 1.1*np.max(u0))

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title("Wave propagation on a vibrating string")

def update(frame):
    line.set_ydata(frame)
    return line,

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=50,
    blit=True,                   # for speed up
    repeat=False
)

plt.show()

---

### Graphic interpretation 

The graphic reveal typical wave behaviour.

Observations:

- The initial **Gaussian bump splits into two travelling waves**.
- Waves propagate in opposite directions.
- At the boundaries they reflect due to the fixed-end condition.

This is exactly the physical behaviour of the **1D wave equation on a string**.

---

### Comparison with the Heat Equation

| Heat equation | Wave equation |
|---|---|
| diffusion | propagation |
| smoothing | oscillation |
| energy dissipates | energy approximately conserved |


---
###  Hidden matrix–vector multiplication in the code

In the update loop we compute the term

$$
u_{i-1} - 2u_i + u_{i+1},
$$

which approximates the second spatial derivative.

Although this is written as a loop over grid points, it can be interpreted as a **matrix–vector multiplication**.

The matrix corresponding to that stencil is 

$$
A =
\begin{bmatrix}
-2 & 1 & 0 & \cdots \\
1 & -2 & 1 & \cdots \\
0 & 1 & -2 & \cdots \\
\vdots & & & \ddots
\end{bmatrix}.
$$

If we collect the interior values into a vector

$$
\mathbf{u} = \begin{bmatrix}
u_1 \\
u_2 \\
\vdots \\
u_{N-1}
\end{bmatrix},
$$

then the operation

$$
u_{i-1} - 2u_i + u_{i+1}
$$

can be written as


$$
A\mathbf{u} =
\begin{bmatrix}
u_2 - 2u_1 \\
u_1 - 2u_2 + u_3 \\
u_2 - 2u_3 + u_4 \\
\vdots \\
\end{bmatrix},
$$


which reproduces the finite difference stencil.

So the update rule in the code is equivalent to

$$
\mathbf{u}^{n+1} = 2\mathbf{u}^n - \mathbf{u}^{n-1} +
\frac{c^2 \Delta t^2}{h^2} A \mathbf{u}^n,
$$
but the loop in the code is performing a **matrix operation  $ A\mathbf{u}   $  in disguise**.

Note: This approach is actually more efficient than  building the matrix explicitly. \
High-performance PDE solvers usually apply the operator matrix-free.



---
## 5. Matrix formulation

In [ ]:
#  What explicit matrix multiplication would look like 


Nx = 100               # like in animation
N = Nx - 1               
A = np.zeros((N, N))    

for i in range(N):
    A[i, i] = -2
    if i > 0:
        A[i, i-1] = 1
    if i < N-1:
        A[i, i+1] = 1

lap = A @ u[1:Nx]

print("Discrete Laplacian matrix 100x100")
print(A)
print()
print("A @ u[1:Nx]")
print(lap)


---

✅ We have now connected:

$$
\text{PDE physics}
\longrightarrow
\text{finite difference stencil}
\longrightarrow
\text{loop over grid poins } 
\longrightarrow
\text{impicit matrix operation}
$$


---
## 6. Eigenmodes of the Discrete Laplacian


👉 **Eigenvectors of the discrete operator correspond to standing wave modes of the system.**


In [ ]:
# Draw the first four eigenmodes  

# Build tridiagonal matrix A (discrete Laplacian)
# Nx = 100 
N = Nx - 1   # number of interior points

A = np.zeros((N, N))

for i in range(N):
    A[i, i] = -2
    if i > 0:
        A[i, i-1] = 1
    if i < N-1:
        A[i, i+1] = 1

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)

# Sort by eigenvalue (smallest magnitude first for nicer ordering)
idx = np.argsort(np.abs(eigenvalues))
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Extend eigenvectors to include boundary points (u=0 at ends)
x_interior = x[1:-1]

# Plot first few eigenvectors
fig, axes = plt.subplots(2, 2, figsize=(8,6))

num_modes = 4

for k, ax in enumerate(axes.flat):
    v = eigenvectors[:, k]
    
    # normalize for nicer plotting
    v = v / np.max(np.abs(v))
    
    # include boundary points
    v_full = np.zeros(Nx+1)
    v_full[1:-1] = v
    
    ax.plot(x, v_full)
    ax.set_title(f"Mode {k+1}")
    ax.set_xlim(0, L)
    ax.set_ylim(-1.1, 1.1)
    ax.grid()

plt.suptitle("Eigenmodes of the discrete Laplacian", fontsize=14)
plt.tight_layout()
plt.show()
print("Count the number of zero crossings in each mode. What pattern do you observe?")

## Eigenvectors as standing waves

In the previous section, we saw that the finite difference operator can be written as a matrix $A$.
We just computed its eigenvalues and eigenvectors using function:

```python
np.linalg.eig(A).
```

An eigenvector $\mathbf{v}$ of $A$ satisfies

$$
A \mathbf{v} = \lambda \mathbf{v},
$$

where $\lambda$ is the corresponding eigenvalue.

Note: $\mathbf{v}$ denotes an eigenvector, not velocity.

---

### What do the eigenvectors look like?

When we plot the first few eigenvectors, we observe that they have the shape of **sine waves**.

This is not a coincidence. In the continuous case, the wave equation has solutions of the form

$$
u(x,t) = \sin(kx)\cos(\omega t),
$$

which describe **standing waves** on a string.

---

### Discrete vs continuous picture

The matrix $A$ is a discrete version of the second derivative operator

$$
\frac{\partial^2}{\partial x^2}.
$$

Its eigenvectors are therefore discrete analogues of the continuous functions

$$
\sin\left(\frac{n\pi x}{L}\right).
$$

Each eigenvector corresponds to a different **mode of vibration**:

* mode 1 → lowest frequency (long wavelength)
* mode 2 → one additional oscillation
* mode 3 → higher frequency
* etc.

---

### Why this matters for the wave equation?

The wave equation evolves according to

$$
\mathbf{u}^{n+1} = 2\mathbf{u}^n - \mathbf{u}^{n-1}+\frac{c^2 \Delta t^2}{h^2} A \mathbf{u}^n.
$$

If the initial condition is an eigenvector of $A$, then the solution evolves **without changing shape**, only oscillating in time.

👉 This means:

* eigenvectors of $A$ are the **natural vibration modes** of the system
* eigenvectors correspond to **standing waves**
* the full solution is a **combination of these modes**


✅  We have now connected:

$$
\text{wave equation}
\longrightarrow
\text{finite differences}
\longrightarrow
\text{matrix } A
\longrightarrow
\text{eigenvectors (standing waves)}
$$

This is one of the key ideas in computational physics:

👉 **linear algebra explains the structure of physical systems**.




---
## 7. Evolution of a Single Eigenmode

👉 **Eigenvectors of the discrete operator correspond to standing wave modes of the system.**

In [ ]:
#  Evolution of a single eigenmode 

%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# --- Physical parameters ---
L = 1.0
T = 1.0
c = 1.0

# --- Numerical parameters ---
Nx = 100
h = L / Nx
dt = 0.5 * h / c
Nt = int(T / dt)

x = np.linspace(0, L, Nx+1)

# --- Build matrix A (discrete Laplacian) ---
N = Nx - 1
A = np.zeros((N, N))

for i in range(N):
    A[i, i] = -2
    if i > 0:
        A[i, i-1] = 1
    if i < N-1:
        A[i, i+1] = 1

# --- Eigen decomposition ---
eigenvalues, eigenvectors = np.linalg.eig(A)

# sort modes
idx = np.argsort(np.abs(eigenvalues))
eigenvectors = eigenvectors[:, idx]

# --- Choose one mode ---
mode_index = 2   # try 0,1,2,... to see different modes
v = eigenvectors[:, mode_index]

# normalize
v = v / np.max(np.abs(v))

# extend to full grid (boundary = 0)
u0 = np.zeros(Nx+1)
u0[1:-1] = v

# zero initial velocity
v0 = np.zeros_like(u0)

# --- Initialize time stepping ---
u_prev = u0.copy()
u = u0 + dt*v0
u_next = np.zeros_like(u)

# --- Store frames ---
frames = [u.copy()]

for n in range(Nt):

    for i in range(1, Nx):
        u_next[i] = (
            2*u[i] - u_prev[i]
            + (c**2 * dt**2 / h**2) * (u[i-1] - 2*u[i] + u[i+1])
        )

    if n % 5 == 0:
        frames.append(u.copy())

    u_prev[:] = u
    u[:] = u_next


# --- Animation ---
fig, ax = plt.subplots()
ax.set_autoscale_on(False)

line, = ax.plot(x, frames[0])

ax.set_xlim(0, L)
ax.set_ylim(-1.1, 1.1)

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title(f"Evolution of eigenmode {mode_index+1}")

def update(frame):
    line.set_ydata(frame)
    return line,

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=50,
    blit=True,
    repeat=False
)

plt.show()

###  Evolution of a single eigenmode

We now choose the initial condition to be a single eigenvector of the matrix $A$:

$$
A \mathbf{v} = \lambda \mathbf{v}.
$$

This vector is used as the initial displacement:

$$
\mathbf{u}^0 = \mathbf{v}, \qquad \mathbf{v}^0 = 0.
$$



###  What happens during the simulation?

The wave equation update rule is

$$
\mathbf{u}^{n+1} = 2\mathbf{u}^n - \mathbf{u}^{n-1} + \frac{c^2 \Delta t^2}{h^2} A \mathbf{u}^n.
$$

If $\mathbf{u}^n$ is proportional to an eigenvector $\mathbf{v}$, then

$$
A \mathbf{u}^n = \lambda \mathbf{u}^n.
$$

So the update becomes
$$
\mathbf{u}^{n+1} = (2 + \frac{c^2 \Delta t^2}{h^2}\lambda)\mathbf{u}^n - \mathbf{u}^{n-1}.
$$



###  Key observation

The solution always remains proportional to the same vector $\mathbf{v}$.

This means:

* the **shape does not change**
* only the **amplitude varies in time**



###  Physical interpretation

This is exactly a **standing wave**:

$$
u(x,t) = \sin\left(\frac{n\pi x}{L}\right)\cos(\omega t).
$$

Each eigenvector corresponds to a **natural vibration mode** of the string.


###  What you should see in the animation

* the curve keeps the same shape
* it oscillates up and down
* no distortion or spreading occurs

This confirms that eigenvectors of $A$ are the **fundamental modes of the system**.


✅ We have demonstrated numerically that

$ \quad \quad 
\mathbf{\text{linear algebra (eigenvectors)}} \;
 \Longleftrightarrow   \;
 \mathbf{\text{physics (standing waves)}}.$


This is one of the key ideas in computational science.

👉 **Each eigenmode evolves independently, so the full solution is a superposition of simple oscillations.**

---
*Heikki Miettinen 2026*